# 02. From raw counts to a normalized matrix: which cells would I throw away?

You start again from the raw counts, as if nobody had touched the data, and redo QC and normalization yourself. The authors' version is your answer key at the end.

**Output:** `C:/bio_data/ra_pbmc_qc.h5ad`

> **How this notebook works.** Each step has a **spec** (what the code must do), a **read-first** link, an empty code cell for you, and a **TEST** cell written by Claude. Your cell is done when its test prints PASS and you can say why your approach works. The ✍️ cells are yours to write; predictions go in *before* you run the cell they predict.
>
> No function names are given on purpose. Find them in the read-first link or the scanpy docs. Stuck for more than 10 minutes on one cell: ask Claude for a *question*, not code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

## Q1. A fresh object from raw counts

**Spec:** load the file into `authors`. Build a **new** AnnData `adata` with: the raw counts as `.X`, a copy of `authors.obs`, and the raw `var` table. Then set gene symbols as `var_names` and make them unique (you counted the duplicates in 01 E3).

**Why new and not edit in place:** you want the authors' version untouched for comparison at Q7.

In [ ]:
# authors = the file as loaded
# adata = new AnnData: raw counts, copied obs, raw var
# var_names = symbols, made unique

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
import scipy.sparse as sp
assert adata.shape == authors.shape
assert adata.var_names.is_unique, "var_names not unique yet"
assert "CD14" in adata.var_names, "var_names are not symbols yet"
d = adata.X.data[:500_000]
assert np.allclose(d, np.round(d)), ".X is not raw counts"
print("PASS:", adata.shape)

## Q2. Flag mitochondrial, ribosomal and hemoglobin genes

**Spec:** three boolean columns in `adata.var`: `mt`, `ribo`, `hb`. Mito genes start with `MT-`, ribosomal with `RPS` or `RPL`, hemoglobin genes match `HB` followed by anything except `P` (HBP1 is not hemoglobin).

**Read first:** https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html, the code that sets up these flags. Why each one: mito = dying cells leak cytoplasm and keep mitochondria, hb = red blood cell contamination.

In [ ]:
# adata.var['mt'], ['ribo'], ['hb'] = boolean masks on var_names

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert adata.var["mt"].sum() == 13, adata.var["mt"].sum()
assert adata.var["ribo"].sum() > 50 and adata.var["hb"].sum() > 0
assert not adata.var.loc["HBP1", "hb"] if "HBP1" in adata.var_names else True
print("PASS: mt", adata.var["mt"].sum(), "ribo", adata.var["ribo"].sum(), "hb", adata.var["hb"].sum())

## Q3. Compute QC metrics yourself

**Spec:** first rename the authors' `pct_counts_mt` in `adata.obs` to `authors_pct_counts_mt` (the next step overwrites that name). Then let scanpy compute per-cell QC metrics for the three gene groups.

In [ ]:
# rename the authors' column
# compute QC metrics for mt, ribo, hb

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
r = np.corrcoef(adata.obs["pct_counts_mt"], adata.obs["authors_pct_counts_mt"])[0, 1]
assert r > 0.99, f"r = {r:.3f}: your mito % does not match the authors'"
print(f"PASS: your mito % matches the authors' (r = {r:.4f})")

## Q4. Look before you cut

**Spec:** one figure, three panels: histograms of `total_counts`, `n_genes_by_counts`, `pct_counts_mt`. Then a scatter of total counts against genes, coloured by mito %. Claim titles, not captions (the 02 rule).

✍️ **Prediction: where will you see hard edges, and where will the dying cells sit in the scatter?**

_..._

In [ ]:
# 3 histograms in one figure + 1 scatter coloured by mito %

...

✍️ **What do you see? Name the edges with numbers.**

_..._

## Q5. Write your own outlier rule

**Spec:** write a function `is_outlier(values, nmads)` that returns True for every value more than `nmads` median absolute deviations away from the median. Write it by hand with numpy or pandas, no library outlier function. This is the rule sc-best-practices recommends instead of fixed cutoffs.

**Read first:** https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html, the MAD section.

In [ ]:
# def is_outlier(values, nmads): ...
#   median, MAD = median of |x - median|, return boolean array

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
out = np.asarray(is_outlier(pd.Series([1, 2, 3, 4, 100]), 5))
assert out.tolist() == [False, False, False, False, True], out
out2 = np.asarray(is_outlier(pd.Series([10, 10, 10, 11, 9]), 3))
assert out2.sum() == 0, "a tight cluster should have no outliers"
print("PASS: is_outlier works on the known cases")

**Spec, part 2:** flag a cell in `adata.obs['outlier']` if it is an outlier (5 MADs) on log1p total counts **or** log1p genes, **or** if its mito % is an outlier at 3 MADs. Store the number flagged in `n_flagged`.

✍️ **Prediction: the authors already filtered with hard cutoffs. Will your MAD rule flag many cells or few? Why?**

_..._

In [ ]:
# adata.obs['outlier'] = combined flag
# n_flagged = how many cells

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert adata.obs["outlier"].dtype == bool
assert n_flagged == adata.obs["outlier"].sum()
assert n_flagged < 0.15 * adata.n_obs, "you are flagging a lot: check your MAD logic"
print(f"PASS: {n_flagged} cells flagged ({100 * n_flagged / adata.n_obs:.1f}%)")

✍️ **Decision: do you remove them? Look at which authors' labels the flagged cells carry first (a crosstab). If they pile up in one cell type, cutting them is cutting biology. Write the decision and the evidence.**

_..._

In [ ]:
# crosstab outlier vs fine_annot, then apply your decision

...

## Q6. Doublets

**Spec:** `demux_doublet_call` (genotype-based) and `scrub_doublets` (expression-based) are the authors' doublet calls. Count both.

In [ ]:
# value counts of the two doublet columns

...

✍️ **What did the authors already remove, and what is left? Do you remove the rest? (Stretch: rerun scrublet yourself per batch and compare.)**

_..._

## Q7. Normalize, and keep the counts

**Spec:** store the raw counts in `adata.layers['counts']` **before** you normalize (05 and 06 need them). Then scale every cell to the same total as the authors (your 01 E2 number) and log1p.

**Read first:** https://www.sc-best-practices.org/preprocessing_visualization/normalization.html

In [ ]:
# layers['counts'] = copy of raw counts
# normalize to the authors' target sum, then log1p

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
c = adata.layers["counts"].data[:500_000]
assert np.allclose(c, np.round(c)), "layers['counts'] is not counts: you normalized before copying"
s = np.asarray(np.expm1(adata.X[:1000]).sum(axis=1)).ravel()
assert np.allclose(s, 10000, rtol=0.01), "rows do not sum to the target after undoing log"
print("PASS: counts kept, X is log1p of", 10000)

## Q8. Check against the answer key

**Spec:** for 200 random cells, correlate your `.X` with the authors' `.X` per cell, then take the median into `median_r`. Remember 01 E2 part 3: compare only on the genes the authors kept, or you are comparing against 192 forced zeros.

**Do not drop those genes from your own object.** Whether to keep mito and ribosomal genes out of clustering is a decision for 03, not a side effect of a check.

In [ ]:
# 200 random cells, genes the authors kept, correlation per cell, median -> median_r

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert median_r > 0.999, f"median r = {median_r:.4f}"
print(f"PASS: your matrix matches the authors' (median r = {{median_r:.4f}})")

## Q9. Save

**Spec:** write `adata` to `C:/bio_data/ra_pbmc_qc.h5ad`. Never into the repo (the `.gitignore` blocks `*.h5ad`, but do not rely on it).

In [ ]:
# write the file

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
import pathlib
p = pathlib.Path("C:/bio_data/ra_pbmc_qc.h5ad")
assert p.exists() and p.stat().st_size > 50e6
print("PASS: saved", round(p.stat().st_size / 1e6), "MB")

## Teach-back

3 to 5 sentences in your own words: what this notebook did, what it found, and one thing that surprised you. No sentence means not learned.

_..._

## AI note

What did Claude write (the specs, the tests), what did you write, and where did you get stuck and ask for a hint?

_..._